# Redação: ajuste fino do BERTimbau, versão 2 (teste com GPU)

Resultado da versão 1 (uma semente, taxa 3e-5, 4 épocas): **QWK médio 0,410** no teste, erro total 111,4, e a melhor época foi a **primeira** (mais treino não ajudou).
Esta versão muda só o que costuma ajudar nesse quadro, **escolhido antes de ver o novo resultado**:
- taxa de aprendizado menor (**2e-5**) e **3 épocas**;
- **3 sementes** treinadas em sequência, com a melhor época de cada uma escolhida pela validação;
- o resultado final é a **média das previsões das 3 sementes** (conjunto), que reduz a variação de uma rodada só.

**Meta fixada antes do teste:** QWK médio do conjunto **igual ou maior que 0,50** e nenhuma competência abaixo de 0,40 para valer investir mais. Abaixo disso, encerramos o treino próprio para nota. **Não se escolhem ajustes olhando o teste.**

**Como usar**
- **Kaggle:** Accelerator GPU T4 x2, Internet ligada, "Run All". Mesmo ambiente da versão 1.
- Leva **de 1 a 3 horas** no total (3 treinos; estimativa, não medida). Deixe a aba aberta.
- No fim imprime a tabela e grava `resultado.json`. Me mande a tabela.

**O que ele usa:** só o corpus público Essay-BR (revisão fixada, checksum conferido) e o modelo aberto `neuralmind/bert-base-portuguese-cased`. Nenhum dado de aluno do Olena entra.

**Limites:** redações de simulado, não de prova; direitos de terceiros dos textos não verificados para uso em produto; isto é pesquisa e não libera nota no app.

In [ ]:
# Configuração. Para testar o caderno rapidamente (CPU, poucos minutos): SMOKE = True.
SMOKE = False
MODEL = "neuralmind/bert-base-portuguese-cased"
MAXLEN = 512
EPOCHS = 3
LR = 2e-5
BATCH = 8
SEEDS = [1, 2, 3]

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "scikit-learn"], check=True)

In [ ]:
import ast, csv, hashlib, json, random, re, time, unicodedata, urllib.request
from collections import defaultdict
from pathlib import Path
import numpy as np
import torch
from sklearn.metrics import cohen_kappa_score
from transformers import AutoModel, AutoTokenizer

REVISION = "31e502ff3075e182cdda2560a767be65db677fa5"
EXPECTED_SHA256 = "f160cd1469df9e68dde0a60c09b2e9865eda76e9bc24b1cc6b610a2d5406ceef"
EXPECTED_SPLITS = {"train": 3122, "validation": 792, "test": 654}
SPLIT_SEED = "olena-essay-prompt-holdout-v1"   # a mesma divisão por tema do restante da pesquisa

work = Path("essay-work"); work.mkdir(exist_ok=True)
csv_path = work / "essay-br.csv"
if not csv_path.exists():
    url = f"https://raw.githubusercontent.com/rafaelanchieta/essay/{REVISION}/essay-br/essay-br.csv"
    urllib.request.urlretrieve(url, csv_path)
digest = hashlib.sha256(csv_path.read_bytes()).hexdigest()
assert digest == EXPECTED_SHA256, f"Corpus diferente do esperado: {digest}"
print("corpus conferido")

In [ ]:
def fingerprint(text):
    words = re.findall(r"\w+", unicodedata.normalize("NFKC", text).casefold())
    return hashlib.sha256(" ".join(words).encode()).hexdigest()

with csv_path.open(encoding="utf-8", newline="") as stream:
    rows = list(csv.DictReader(stream))

buckets, invalid = defaultdict(list), 0
for index, row in enumerate(rows):
    try:
        paragraphs = ast.literal_eval(row["essay"])
        grades = ast.literal_eval(row["competence"])
        assert isinstance(paragraphs, list) and all(isinstance(p, str) for p in paragraphs)
        assert len(grades) == 5 and all(type(g) is int and g in range(0, 201, 40) for g in grades)
        assert sum(grades) == int(row["score"])
        text = "\n\n".join(p.strip() for p in paragraphs if p.strip())
        prompt = unicodedata.normalize("NFKC", row["prompt"]).strip().casefold()
        assert text and prompt
    except (AssertionError, ValueError, SyntaxError, TypeError, KeyError):
        invalid += 1
        continue
    buckets[fingerprint(text)].append({"text": text, "prompt": prompt, "grades": grades})

kept = []
for candidates in buckets.values():
    if len({(c["prompt"], tuple(c["grades"])) for c in candidates}) != 1:
        continue  # notas ou temas em conflito: excluídos, não misturados
    kept.append(candidates[0])

prompts = sorted({r["prompt"] for r in kept}, key=lambda p: hashlib.sha256(f"{SPLIT_SEED}:{p}".encode()).hexdigest())
train_end, validation_end = round(len(prompts) * .7), round(len(prompts) * .85)
assign = {p: "train" if i < train_end else "validation" if i < validation_end else "test" for i, p in enumerate(prompts)}
for r in kept:
    r["split"] = assign[r["prompt"]]
train = [r for r in kept if r["split"] == "train"]
val = [r for r in kept if r["split"] == "validation"]
test = [r for r in kept if r["split"] == "test"]
counts = {"train": len(train), "validation": len(val), "test": len(test)}
print(counts)
assert counts == EXPECTED_SPLITS, "A divisão não bate com a da pesquisa; não compare com os resultados anteriores."
assert not {r["prompt"] for r in train} & {r["prompt"] for r in test}  # temas separados

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("dispositivo:", device)
if device == "cpu" and not SMOKE:
    print("ATENÇÃO: sem GPU isto levará muitas horas. Ative a GPU nas configurações.")
if SMOKE:
    train, val, test, EPOCHS, MAXLEN, SEEDS = train[:32], val[:16], test[:16], 1, 128, SEEDS[:2]

tok = AutoTokenizer.from_pretrained(MODEL)

class Grader(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = AutoModel.from_pretrained(MODEL)
        self.head = torch.nn.Linear(self.enc.config.hidden_size, 5)
    def forward(self, **batch):
        hidden = self.enc(**batch).last_hidden_state
        mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        return self.head((hidden * mask).sum(1) / mask.sum(1))

def batches(data, size, shuffle):
    order = list(range(len(data)))
    if shuffle: random.shuffle(order)
    for i in range(0, len(order), size):
        chunk = [data[j] for j in order[i:i + size]]
        enc = tok([c["text"] for c in chunk], truncation=True, max_length=MAXLEN, padding=True, return_tensors="pt")
        target = torch.tensor([[g / 200 for g in c["grades"]] for c in chunk], dtype=torch.float)
        yield {k: v.to(device) for k, v in enc.items()}, target.to(device)

def predict(model, data):
    model.eval(); out = []
    with torch.no_grad():
        for enc, _ in batches(data, 16, False):
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=device == "cuda"):
                out.append(model(**enc).float().cpu().numpy())
    return np.clip(np.concatenate(out) * 200, 0, 200)

def snap(p): return (np.clip(np.round(p / 40), 0, 5) * 40).astype(int)

def metrics(pred, data):
    y = np.array([r["grades"] for r in data]); p = snap(pred)
    qwk = [float(cohen_kappa_score(y[:, k] // 40, p[:, k] // 40, weights="quadratic")) for k in range(5)]
    return {"qwk": qwk, "qwk_medio": float(np.mean(qwk)),
            "mae_competencia": [float(m) for m in np.abs(y - p).mean(0)],
            "mae_total": float(np.abs(y.sum(1) - p.sum(1)).mean())}

In [ ]:
def run(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = Grader().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * ((len(train) + BATCH - 1) // BATCH)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda s: min(1.0, (s + 1) / (0.1 * steps)) * max(0.0, (steps - s) / steps))
    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")
    history, best_score, best_state, best_epoch, step, t0 = [], -9.0, None, 0, 0, time.time()
    for epoch in range(EPOCHS):
        model.train()
        for enc, target in batches(train, BATCH, True):
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=device == "cuda"):
                loss = torch.nn.functional.mse_loss(model(**enc).float(), target)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step(); optimizer.zero_grad()
            step += 1
            if step % 50 == 0:
                print(f"semente {seed} época {epoch + 1}/{EPOCHS} passo {step}/{steps} perda {loss.item():.4f} {time.time() - t0:.0f}s", flush=True)
        m = metrics(predict(model, val), val)
        history.append({"epoca": epoch + 1, **m})
        print(f"semente {seed} validação, época {epoch + 1}: QWK médio {m['qwk_medio']:.3f} | MAE total {m['mae_total']:.1f}", flush=True)
        if m["qwk_medio"] > best_score:   # a época é escolhida SÓ pela validação
            best_score, best_epoch = m["qwk_medio"], epoch + 1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    result = {"seed": seed, "best_epoch": best_epoch, "history": history,
              "val_pred": predict(model, val), "test_pred": predict(model, test)}
    del model, optimizer, best_state
    if device == "cuda": torch.cuda.empty_cache()
    return result

runs = [run(seed) for seed in SEEDS]
print("sementes treinadas:", [(r["seed"], r["best_epoch"]) for r in runs])

In [ ]:
median = np.median([r["grades"] for r in train], axis=0)
baseline = metrics(np.tile(median, (len(test), 1)), test)
singles = [metrics(r["test_pred"], test) for r in runs]
ensemble_val = metrics(np.mean([r["val_pred"] for r in runs], axis=0), val)
ensemble = metrics(np.mean([r["test_pred"] for r in runs], axis=0), test)

previous = {  # rodadas anteriores, mesma divisão (docs/research/ESSAY_OWN_GRADER.md)
    "Estrutura (3.122 redações)": ([0.150, 0.362, 0.356, 0.358, 0.383], 115.84),
    "Semântico congelado + estrutura (3.122)": ([0.293, 0.422, 0.279, 0.373, 0.389], 119.51),
    "BERTimbau ajustado, CPU (400 redações)": ([0.254, 0.370, 0.247, 0.323, 0.304], 125.02),
    "BERTimbau, GPU, versão 1 (1 semente, 3e-5)": ([0.336, 0.485, 0.363, 0.457, 0.410], 111.4),
}
line = lambda name, q, mae: f"| {name} | " + " | ".join(f"{x:.3f}" for x in q) + f" | {sum(q) / 5:.3f} | {mae:.1f} |"
print("| Modelo | C1 | C2 | C3 | C4 | C5 | QWK médio | MAE total |")
print("|---|---:|---:|---:|---:|---:|---:|---:|")
print(line("Mediana do treino", baseline["qwk"], baseline["mae_total"]))
for name, (q, mae) in previous.items():
    print(line(name, q, mae))
for r, m in zip(runs, singles):
    print(line(f"Versão 2, semente {r['seed']} (época {r['best_epoch']})", m["qwk"], m["mae_total"]))
print(line("**Versão 2, conjunto das sementes**", ensemble["qwk"], ensemble["mae_total"]))
print(f"\nValidação do conjunto: QWK médio {ensemble_val['qwk_medio']:.3f} (se for bem maior que o do teste, há decoreba dos temas).")

ok = ensemble["qwk_medio"] >= 0.50 and min(ensemble["qwk"]) >= 0.40
print("\nMETA (QWK médio >= 0,50 e nenhuma competência < 0,40):", "ATINGIDA" if ok else "NÃO atingida")
print("A literatura reporta QWK de 0,60 a 0,73 por competência (outro corpus/divisão, não comparável diretamente).")

Path("resultado.json").write_text(json.dumps({
    "modelo": MODEL, "redacoes_treino": len(train), "max_tokens": MAXLEN, "epocas": EPOCHS, "lr": LR,
    "sementes": SEEDS, "dispositivo": device, "smoke": SMOKE, "meta_atingida": bool(ok),
    "teste_conjunto": ensemble, "teste_por_semente": singles, "validacao_conjunto": ensemble_val,
    "mediana": baseline, "historico": [{"seed": r["seed"], "melhor_epoca": r["best_epoch"], "validacao": r["history"]} for r in runs],
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("gravado: resultado.json (sem texto de redação)")